**Fine-Tuning BERT for POS Tagging &** **Chunking**

Task 1: Dataset Selection

Dataset Chosen:

Universal Dependencies (UD) → for POS Tagging
CoNLL-2003 → for Chunking (NER-style chunk labels)

Label Categories:

POS Tags (UD example)
NOUN, VERB, ADJ, ADV, PRON, DET, ADP, AUX, CCONJ, NUM, PART, INTJ, PROPN, PUNCT
Chunk Tags (CoNLL format)
B-NP (Beginning of Noun Phrase)
I-NP (Inside Noun Phrase)
B-VP (Verb Phrase)
I-VP
B-PP (Prepositional Phrase)
O (Outside)

Task 2: Data Preprocessing

Key Steps:
Tokenization using BERT tokenizer

Align labels with subwords

Assign -100 for ignored token

In [29]:
#install
!pip install transformers datasets seqeval

In [30]:
#Import Libraries
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

In [31]:
#Create Dataset
data = [
    {"tokens": ["John", "works", "at", "Google"], "labels": ["NOUN", "VERB", "ADP", "NOUN"]},
    {"tokens": ["Mary", "lives", "in", "London"], "labels": ["NOUN", "VERB", "ADP", "NOUN"]},
    {"tokens": ["He", "is", "a", "doctor"], "labels": ["PRON", "VERB", "DET", "NOUN"]},
]

dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['tokens', 'labels'],
    num_rows: 3
})


In [32]:
#Label Mapping
label_list = list(set(label for row in data for label in row["labels"]))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [33]:
#Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [34]:
#Tokenization + Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Task 3: Model Setup

In [35]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Task 4: Training

In [38]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs",
)
from datasets import load_metric
metric = load_metric("seqeval")
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/datasets/load.py:759: FutureWarning: The repository for seqeval contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/seqeval/seqeval.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


Task 5: Evaluation

In [41]:
trainer.train()
trainer.predict(tokenized_dataset)

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


PredictionOutput(predictions=array([[[ 0.6746432 ,  0.36506167,  0.3294293 ,  0.17827135,
         -0.01779787],
        [ 1.3590556 , -0.10350434, -0.22862951, -0.5263057 ,
         -0.34673223],
        [ 0.41512722,  1.2453613 ,  0.00969055, -0.9711137 ,
         -0.32620972],
        [ 0.327999  ,  0.07649635,  0.7105583 , -0.78231084,
         -0.39193958],
        [ 1.6704471 ,  0.01420657, -0.08695164, -0.819775  ,
         -0.40466952],
        [ 0.56504107,  0.37486124,  0.29185924,  0.22194418,
         -0.31842747]],

       [[ 0.5952142 ,  0.23271023,  0.23540458, -0.04320502,
         -0.24606264],
        [ 1.2841992 , -0.0497613 , -0.6389885 , -0.7521904 ,
         -0.64649564],
        [ 0.24142146,  1.226265  , -0.18007898, -0.9239551 ,
         -0.3648268 ],
        [ 0.18457974,  0.22477137,  1.2549903 , -1.0958143 ,
         -0.20528625],
        [ 1.9569471 ,  0.14318885, -0.5644983 , -1.0338284 ,
         -0.47449076],
        [ 0.7208129 ,  0.11229621,  0.4206088

Task 6: Inference

In [42]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"
result = nlp(sentence)

for r in result:
    print(r)

{'entity': 'NOUN', 'score': 0.58282214, 'index': 1, 'word': 'John', 'start': 0, 'end': 4}
{'entity': 'VERB', 'score': 0.4674971, 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'ADP', 'score': 0.32493043, 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'NOUN', 'score': 0.4784449, 'index': 4, 'word': 'Google', 'start': 14, 'end': 20}
{'entity': 'ADP', 'score': 0.3072136, 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NOUN', 'score': 0.55409735, 'index': 6, 'word': 'California', 'start': 24, 'end': 34}
